<a href="https://colab.research.google.com/github/kshitijtandon/fisabio-gapseq-workshop/blob/main/notebooks/FISABIO_gapseq_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FISABIO gapseq workshop

This is the live demo of how to develop a genome-scale metabolic model of a bacterial genome or Metagenome-assembled-genome using [gapseq](https://link.springer.com/article/10.1186/s13059-021-02295-1$0).

I have created a [GitHub]() repo for ease of analysis. As generating a GEM can take several hours. So, you will have all the files ready for you view while we go through the steps.




In [4]:
# Clone the workshop repository

!git clone https://github.com/kshitijtandon/fisabio-gapseq-workshop.git

%cd fisabio-gapseq-workshop

Cloning into 'fisabio-gapseq-workshop'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 80 (delta 26), reused 43 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (80/80), 6.55 MiB | 14.15 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/fisabio-gapseq-workshop


In [5]:
%%bash

echo "Repository contents:"

ls -lh

echo ""

echo "Data files:"

ls -lh data

Repository contents:
total 20K
drwxr-xr-x 2 root root 4.0K Jun 29 03:49 data
-rw-r--r-- 1 root root 1.1K Jun 29 03:49 LICENSE
drwxr-xr-x 2 root root 4.0K Jun 29 03:49 notebooks
-rw-r--r-- 1 root root 1.7K Jun 29 03:49 README.md
drwxr-xr-x 2 root root 4.0K Jun 29 03:49 scripts

Data files:
total 39M
-rw-r--r-- 1 root root 173K Jun 29 03:49 ecoli_k12-all-Pathways.tbl
-rw-r--r-- 1 root root 8.6M Jun 29 03:49 ecoli_k12-all-Reactions.tbl
-rw-r--r-- 1 root root 801K Jun 29 03:49 ecoli_k12-draft.RDS
-rw-r--r-- 1 root root  11M Jun 29 03:49 ecoli_k12-draft.xml
-rw-r--r-- 1 root root 881K Jun 29 03:49 ecoli_k12.faa.gz
-rw-r--r-- 1 root root 1.4M Jun 29 03:49 ecoli_k12_genome.fna.gz
-rw-r--r-- 1 root root 4.4M Jun 29 03:49 ecoli_k12-memote.html
-rw-r--r-- 1 root root 810K Jun 29 03:49 ecoli_k12.RDS
-rw-r--r-- 1 root root 134K Jun 29 03:49 ecoli_k12-rxnWeights.RDS
-rw-r--r-- 1 root root 304K Jun 29 03:49 ecoli_k12-rxnXgenes.RDS
-rw-r--r-- 1 root root 642K Jun 29 03:49 ecoli_k12-Transporter.tbl
-r

# Genome-scale metabolic modelling workflow

🧬 Protein FASTA

⬇️

🔍 **find**

• Predict reactions

• Predict pathways

⬇️

🚪 **find-transporters**

• Predict transporters

⬇️

🏗️ **develop a draft**

• Build draft metabolic model

⬇️

🩹 **filling missing elements**

• Fill missing reactions

⬇️

🧫 **Genome-scale metabolic model**

## **Gapseq workflow to create a genome-scale metabolic model**

| Step | Command | Output |
|------|---------|--------|
| 1 | `gapseq find` | Reaction predictions, pathway predictions |
| 2 | `gapseq find-transport` | Transporter predictions |
| 3 | `gapseq draft` | Draft metabolic model |
| 4 | `gapseq fill` | Gap-filled metabolic model |

## **Step 1: Input protein sequences**

gapseq starts from protein sequences predicted from a genome assembly.

For this workshop we use:

**Escherichia coli K-12**

In [6]:
import gzip

n_proteins = 0
protein_lengths = []

with gzip.open("data/ecoli_k12.faa.gz", "rt") as f:
    seq = ""
    for line in f:
        if line.startswith(">"):
            if seq:
                protein_lengths.append(len(seq))
            n_proteins += 1
            seq = ""
        else:
            seq += line.strip()
    if seq:
        protein_lengths.append(len(seq))

print(f"Proteins: {n_proteins:,}")
print(f"Average protein length: {sum(protein_lengths)/len(protein_lengths):.1f} aa")
print(f"Longest protein: {max(protein_lengths):,} aa")

Proteins: 4,298
Average protein length: 309.4 aa
Longest protein: 2,358 aa


## **Step 2: Predict reactions and pathways**

The first major gapseq step is:

```bash
gapseq find -p all ecoli_k12.faa.gz
```

This command searches the predicted protein sequences against curated metabolic databases and maps genes to metabolic functions.

For the workshop, we have already run this step because it can take some time to complete. The command generates several output files that will be used in the subsequent steps of the workflow.

| **Output file** | **Description** |
|-----------------|-----------------|
| `ecoli_k12-all-Reactions.tbl` | Predicted metabolic reactions identified from the protein sequences. |
| `ecoli_k12-all-Pathways.tbl` | Predicted metabolic pathways reconstructed from the identified reactions. |
| `ecoli_k12-rxnWeights.RDS` | Confidence scores assigned to each predicted reaction. These scores are used during the gap-filling step to prioritise high-confidence reactions. |
| `ecoli_k12-rxnXgenes.RDS` | Mapping between predicted reactions and the genes supporting each reaction. |

In [7]:
import pandas as pd

reactions = pd.read_csv(
    "data/ecoli_k12-all-Reactions.tbl",
    sep="\t",
    comment="#",
    engine="python"
)

pathways = pd.read_csv(
    "data/ecoli_k12-all-Pathways.tbl",
    sep="\t",
    comment="#",
    engine="python"
)

Let's take a look at some predicted reactions and pathways

In [8]:
display(reactions.head(5))

,rxn,name,ec,bihit,qseqid,pident,evalue,bitscore,qcovs,stitle,sstart,send,pathway,status,pathway.status,dbhit,complex,exception,complex.status
0,DHLAXANAU-RXN,"1,2-dichloroethane dehalogenase",3.8.1.5,NaN,UniRef90_P9WMS1,25.085,1.130000e-11,60.8,96.0,"NP_414883.5 2-hydroxy-6-ketonona-2,4-dienedioa...",5.0,283.0,|12DICHLORETHDEG-PWY|,bad_blast,NaN,rxn03596 rxn03668 rxn03669 rxn03670 rxn03671 r...,NaN,0,NaN
1,MOXXANAU-RXN,2-chloroethanol dehydrogenase,1.1.2.7,NaN,UniRef90_P38539,26.614,1.230000e-40,154.0,89.0,NP_414666.1 quinoprotein glucose dehydrogenase...,168.0,768.0,|12DICHLORETHDEG-PWY|,bad_blast,NaN,rxn00847 rxn01796 rxn12745 rxn14207 rxn15989 r...,Subunit 1,0,NaN
2,MOXXANAU-RXN,2-chloroethanol dehydrogenase,1.1.2.7,NaN,UniRef90_Q09053,41.176,1.200000e+00,20.8,94.0,NP_415294.1 putative kinase inhibitor [Escheri...,25.0,41.0,|12DICHLORETHDEG-PWY|,bad_blast,NaN,rxn00847 rxn01796 rxn12745 rxn14207 rxn15989 r...,Subunit 2,0,NaN
3,ALDXANAU-RXN,chloroacetaldehyde dehydrogenase,1.2.1.4,NaN,UniRef90_P37685,100.000,0.000000e+00,1065.0,100.0,NP_418045.4 aldehyde dehydrogenase B [Escheric...,1.0,512.0,|12DICHLORETHDEG-PWY|,good_blast,NaN,rxn00507 rxn03455 rxn05880 rxn14071 rxn19116,NaN,0,1.0
4,ALDXANAU-RXN,chloroacetaldehyde dehydrogenase,1.2.1.4,NaN,UniRef90_P37685,40.333,1.440000e-110,334.0,93.0,NP_415816.1 gamma-glutamyl-gamma-aminobutyrald...,23.0,492.0,|12DICHLORETHDEG-PWY|,good_blast,NaN,rxn00507 rxn03455 rxn05880 rxn14071 rxn19116,NaN,0,1.0


### Reaction predictions

Each row represents a predicted metabolic reaction supported by protein-level evidence from the genome.

These reaction predictions are one of the main inputs used to construct the draft metabolic model.

In [9]:
display(pathways.head(5))

,ID,Name,Prediction,Completeness,VagueReactions,KeyReactions,KeyReactionsFound,ReactionsFound
0,|12DICHLORETHDEG-PWY|,"1,2-dichloroethane degradation",False,25,0,1,0,ALDXANAU-RXN
1,|14DICHLORBENZDEG-PWY|,"1,4-dichlorobenzene degradation",False,42,0,2,1,CHLOROBENDDH-RXN DIENELACHYDRO-RXN MALEYLACETA...
2,|1CMET2-PWY|,folate transformations III (E. coli),True,100,0,0,0,DIHYDROFOLATEREDUCT-RXN 5-FORMYL-THF-CYCLO-LIG...
3,|2AMINOBENZDEG-PWY|,anthranilate degradation III (anaerobic),False,0,0,0,0,NaN
4,|2ASDEG-PWY|,orthanilate degradation,False,25,1,0,0,NaN


### Pathway predictions

Pathway predictions summarise the metabolic capabilities inferred from the collection of predicted reactions.

This helps move from individual enzyme/reaction evidence to broader biological functions.

## *How many reactions and pathways were predicted?*

In [ ]:
print(f"Predicted reactions : {len(reactions):,}")
print(f"Predicted pathways  : {len(pathways):,}")

Predicted reactions : 31,988
Predicted pathways  : 1,921


# **Step 3: Predict Transporters**

Transporters determine which metabolites can enter or leave the cell.

They are essential because metabolic models require exchange reactions between the organism and its environment.

Transporters are predicted using:

```bash
gapseq find-transport ecoli_k12.faa.gz
```

In [ ]:
transporters = pd.read_csv(
    "data/ecoli_k12-Transporter.tbl",
    sep="\t",
    comment="#",
    engine="python"
)

print(transporters.shape)

display(transporters.head())

# **Step 4: Construct a draft metabolic model**

The predicted `reactions`, `pathways` and `transporters` are combined into a **draft genome-scale metabolic model**.

This is performed using:

```bash
gapseq draft -r ecoli_k12-all-Reactions.tbl
-t ecoli_k12-Transporter.tbl
-p ecoli_k12-all-Pathways.tbl
```

## *What did gapseq draft actually create?*

In [ ]:
!pip -q install cobra
import logging
import warnings

# Suppress Python warnings
warnings.filterwarnings("ignore")

# Suppress COBRA SBML parser warnings
logging.getLogger("cobra.io.sbml").setLevel(logging.ERROR)

import cobra

draft = cobra.io.read_sbml_model("data/ecoli_k12-draft.xml")

In [ ]:
print(draft)
print(f"Genes:        {len(draft.genes):,}")
print(f"Reactions:    {len(draft.reactions):,}")
print(f"Metabolites:  {len(draft.metabolites):,}")

ecoli_k12
Genes:        1,144
Reactions:    2,802
Metabolites:  2,265


In [ ]:
for rxn in list(draft.reactions)[:5]:
    print(f"{rxn.id}")
    print(f"Name: {rxn.name}")
    print(f"Equation: {rxn.build_reaction_string(use_metabolite_names=True)}")
    print()

## **Do you think this model is ready to simulate growth?**

**Probably No**

## Is the draft model complete?

Although the draft model contains reactions supported by genomic evidence, it may still be **unable to simulate growth**.

Common reasons include:

- Missing or fragmented genes

- Incomplete genome annotations

- Missing reactions in reference databases

- Biological pathways that are only partially reconstructed

Consequently, essential metabolic pathways may remain incomplete, preventing the model from producing biomass.

## **Growth medium**

Before testing whether a model can grow, we must define which nutrients are available.

This information is provided in a **growth medium**.

For this workshop we use a minimal glucose medium:

`MM_glu.csv`

The medium specifies which metabolites are available for uptake from the environment.

In [ ]:
import pandas as pd

medium = pd.read_csv("data/MM_glu.csv")

display(medium)

## Exchange reactions

Exchange reactions connect the metabolic model to its surrounding environment.

They allow nutrients defined in the growth medium to enter the model and metabolic by-products to leave it.

Without exchange reactions, the organism would have no access to external nutrients.

During gap-filling (next step), the available exchange reactions are constrained by the specified growth medium (MM_glu.csv).

Let's see how many and which exchange reactions are present in our draft model.


In [ ]:
exchange_rxns = [r for r in draft.reactions if r.id.startswith("EX_")]

print(f"Exchange reactions: {len(exchange_rxns)}")

Exchange reactions: 207


In [ ]:
for rxn in exchange_rxns[:15]:

    print(f"{rxn.id:20s} {rxn.build_reaction_string(use_metabolite_names=True)}")

# **Step 5: Gap-filling the draft model**

At this stage, we have reconstructed a **draft genome-scale metabolic model** directly from the genome sequence.

However, a draft model is rarely complete.

Why?

- Not every gene can be confidently annotated.
- Some metabolic pathways may be incomplete due to missing or poorly annotated enzymes.
- Genome assemblies may contain gaps or fragmented genes.
- Biological knowledge in reference databases is itself incomplete.

As a result, the draft model may **not be able to produce biomass**, even for organisms that are known to grow experimentally.

To address this, **gapseq performs gap-filling**.

Gap-filling identifies the **smallest set of additional reactions** required to restore growth under a specified growth medium while preferentially retaining reactions that have genomic support.

In this example, we use the minimal glucose medium (`MM_glu.csv`).

The command used is:

```bash
gapseq fill \
    -m ecoli_k12-draft.RDS \
    -n MM_glu.csv \
    -c ecoli_k12-rxnWeights.RDS \
    -g ecoli_k12-rxnXgenes.RDS \
    -b 100
```

The additional input files generated by `gapseq find` help guide the gap-filling process:

| File | Role during gap-filling |
|------|-------------------------|
| `ecoli_k12-rxnWeights.RDS` | Assigns confidence scores to reactions, favouring those supported by genomic evidence. |
| `ecoli_k12-rxnXgenes.RDS` | Preserves gene–reaction associations when selecting candidate reactions. |
| `MM_glu.csv` | Defines the nutrients available for growth. |

#**Step 6: Inspecting the gap-filled model**

Gap-filling produces a **functional genome-scale metabolic model** that can now be used for downstream analyses.

Let's load the gap-filled model and explore some of its key properties.

In [ ]:
import cobra

model = cobra.io.read_sbml_model("data/ecoli_k12.xml")

print("Model summary")
print("-" * 40)
print(f"Genes:        {len(model.genes):,}")
print(f"Reactions:    {len(model.reactions):,}")
print(f"Metabolites:  {len(model.metabolites):,}")

Model summary
----------------------------------------
Genes:        1,144
Reactions:    2,853
Metabolites:  2,278


Our model is now a mathematical representation of E. coli metabolism. It contains the genes, reactions and metabolites required to simulate metabolism under defined environmental conditions.

# **Step 7 Compare draft vs gap-filled model**

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Draft model": [
        len(draft.genes),
        len(draft.reactions),
        len(draft.metabolites)
    ],
    "Gap-filled model": [
        len(model.genes),
        len(model.reactions),
        len(model.metabolites)
    ]
},

index=["Genes", "Reactions", "Metabolites"])
comparison

Notice that only a relatively small number of reactions are typically added during gap-filling, but these additions can be sufficient to restore growth

# **Step 8 Validate the reconstructed model using MEMOTE**

**What is MEMOTE?**

Once a genome-scale metabolic model has been reconstructed, it is good practice to evaluate its quality before performing simulations.

[MEMOTE (Metabolic Model Tests)](https://www.nature.com/articles/s41587-020-0446-y$0) is a community-standard benchmarking tool that automatically assesses the quality of genome-scale metabolic models.

**Rather than determining whether a model is simply “correct” or “incorrect”**, MEMOTE evaluates:

* Structural consistency
* Annotation completeness
* Stoichiometric correctness
* Biomass formulation
* SBML compliance
* Overall model quality

This provides confidence that the reconstructed model is suitable for downstream analyses


**Optional: Generate your own MEMOTE report**

*pip install memote*

*memote report snapshot ecoli_k12.xml --filename ecoli_k12-memote.html*

In [12]:
from IPython.display import HTML

HTML("""
<h3>MEMOTE Validation Report</h3>

<p>
Open the interactive MEMOTE report here:
</p>

<a href="https://kshitijtandon.github.io/fisabio-gapseq-workshop/ecoli_k12-memote.html"
target="_blank">

📊 Open MEMOTE Report

</a>
""")

## What's next?

Once a functional genome-scale metabolic model has been reconstructed, it can be used to address a wide range of biological questions.

For example:

- Can the organism grow on a given nutrient source?

- Which nutrients are essential for growth?

- Which metabolic pathways carry flux?

- What is the effect of knocking out a gene or reaction?

- How do multiple microbial species interact within a community?

These questions are typically addressed using **Flux Balance Analysis (FBA)**.

# Workshop summary

In this session, we have reconstructed a genome-scale metabolic model from a bacterial genome using gapseq.

The workflow consisted of:

1. Protein sequence input
2. Reaction and pathway prediction (`gapseq find`)
3. Transporter prediction (`gapseq find-transport`)
4. Draft model reconstruction (`gapseq draft`)
5. Gap-filling under a defined growth medium (`gapseq fill`)

The resulting model is now ready for downstream metabolic analyses.